# CNN for Blind Image Quality Assessment (IQA)
Predicts quality of rescaled images (0-100 scale) by comparing them with originals

## 1. Imports & Setup

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# Path configuration
DATA_DIR = r'/media/myDisk/1K1K_po_zmianie_rozdzielczosci'  # Update this path
PATCH_SIZE = 32
QUALITY_SCALE_MIN = 0
QUALITY_SCALE_MAX = 100

# Hyperparameters (randomly chosen)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.5
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"📁 Data directory: {DATA_DIR}")
print(f"🔧 Patch size: {PATCH_SIZE}×{PATCH_SIZE}")
print(f"📊 Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}")
print(f"🎲 Learning rate: {LEARNING_RATE}, Dropout: {DROPOUT_RATE}")

## 3. Data Loading & Preprocessing

In [ ]:
def load_image_pair(orig_path, rescaled_path, patch_size=32):
    """
    Load original and rescaled image pair, return as stacked input (patch_size, patch_size, 6)
    """
    try:
        orig_img = Image.open(orig_path).convert('RGB')
        rescaled_img = Image.open(rescaled_path).convert('RGB')
        
        # Resize to patch size
        orig_img = orig_img.resize((patch_size, patch_size), Image.Resampling.LANCZOS)
        rescaled_img = rescaled_img.resize((patch_size, patch_size), Image.Resampling.LANCZOS)
        
        # Convert to numpy and normalize to [0, 1]
        orig_arr = np.array(orig_img, dtype=np.float32) / 255.0
        rescaled_arr = np.array(rescaled_img, dtype=np.float32) / 255.0
        
        # Stack along channel dimension: (32, 32, 6)
        paired = np.concatenate([orig_arr, rescaled_arr], axis=2)
        
        return paired
    except Exception as e:
        print(f"Error loading images: {e}")
        return None


def extract_quality_score(json_metadata):
    """
    Extract quality score from JSON metadata.
    For now: assign 0-100 randomly; can be updated with real metrics (PSNR, SSIM, etc.)
    """
    # TODO: Replace with actual quality metric calculation
    # Could use: PSNR, SSIM, BRISQUE, or custom metric
    return np.random.uniform(20, 95)  # Random quality score for demo


def load_dataset(data_dir, patch_size=32):
    """
    Load all image pairs and their quality scores from JSON metadata
    """
    images = []
    quality_scores = []
    
    data_path = Path(data_dir)
    
    # Find all JSON metadata files
    json_files = sorted(data_path.glob('*.json'))
    
    print(f"Found {len(json_files)} JSON metadata files")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                meta = json.load(f)
            
            # Extract filename info from metadata
            img_num = meta.get('numer_zdjecia')
            img_type = meta.get('typ')
            
            # Find the paired images
            # Assume: id_orig.png and id_*.png exist
            orig_name = f"{img_num}_orig.png"
            rescaled_name = meta.get('nazwa_pliku_img')
            
            orig_path = data_path / orig_name
            rescaled_path = data_path / rescaled_name
            
            if orig_path.exists() and rescaled_path.exists():
                # Load image pair
                paired = load_image_pair(str(orig_path), str(rescaled_path), patch_size)
                
                if paired is not None:
                    images.append(paired)
                    
                    # Extract or calculate quality score
                    quality = extract_quality_score(meta)
                    quality_scores.append(quality)
        
        except Exception as e:
            print(f"Skipping {json_file}: {e}")
            continue
    
    if len(images) > 0:
        X = np.array(images, dtype=np.float32)
        y = np.array(quality_scores, dtype=np.float32)
        print(f"✅ Loaded {len(X)} image pairs")
        print(f"   Shape: {X.shape}")
        print(f"   Quality scores range: [{y.min():.2f}, {y.max():.2f}]")
        return X, y
    else:
        print("❌ No image pairs loaded!")
        return None, None


# Load dataset
X, y = load_dataset(DATA_DIR, PATCH_SIZE)

if X is not None:
    # Normalize quality scores to [0, 1] for training (will denormalize later)
    scaler = MinMaxScaler(feature_range=(0, 1))
    y_normalized = scaler.fit_transform(y.reshape(-1, 1)).flatten()
    
    print(f"✅ Quality scores normalized: [{y_normalized.min():.4f}, {y_normalized.max():.4f}]")

## 4. Build CNN Architecture

In [ ]:
def build_cnn_iqa_model(input_shape=(32, 32, 6), dropout_rate=0.5):
    """
    Build CNN IQA model matching the specified architecture.
    Input: 32×32×6 (original RGB + rescaled RGB stacked)
    Output: 1 (quality score)
    """
    model = models.Sequential([
        # Input: 32×32×6
        layers.Input(shape=input_shape),
        
        # Conv1: 32×32×32
        layers.Conv2D(32, kernel_size=3, strides=1, padding='same', activation='relu'),
        layers.BatchNormalization(),
        
        # MaxPool1: 16×16×32
        layers.MaxPooling2D(pool_size=2, strides=2, padding='valid'),
        
        # Conv2: 16×16×64
        layers.Conv2D(64, kernel_size=3, strides=1, padding='same', activation='relu'),
        layers.BatchNormalization(),
        
        # MaxPool2: 8×8×64
        layers.MaxPooling2D(pool_size=2, strides=2, padding='valid'),
        
        # Conv3: 8×8×128
        layers.Conv2D(128, kernel_size=3, strides=1, padding='same', activation='relu'),
        layers.BatchNormalization(),
        
        # MaxPool3: 4×4×128
        layers.MaxPooling2D(pool_size=2, strides=2, padding='valid'),
        
        # Conv4: 4×4×256
        layers.Conv2D(256, kernel_size=3, strides=1, padding='same', activation='relu'),
        layers.BatchNormalization(),
        
        # MaxPool4: 2×2×256
        layers.MaxPooling2D(pool_size=2, strides=2, padding='valid'),
        
        # Flatten: 1024×1
        layers.Flatten(),
        
        # Dense1: 512×1
        layers.Dense(512, activation='relu'),
        
        # Dropout: rate = 0.5
        layers.Dropout(dropout_rate),
        
        # Output: 1×1 (quality score)
        layers.Dense(1, activation='sigmoid')  # sigmoid for [0,1] output
    ])
    
    return model


# Build model
model = build_cnn_iqa_model(input_shape=(32, 32, 6), dropout_rate=DROPOUT_RATE)

# Display model architecture
model.summary()

## 5. Compile & Train Model

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='mse',
    metrics=['mae', 'mse']
)

print("✅ Model compiled")
print(f"   Optimizer: Adam (lr={LEARNING_RATE})")
print(f"   Loss: MSE")
print(f"   Metrics: MAE, MSE")

In [ ]:
if X is not None and len(X) > 0:
    # Split data: train (80%) -> train (70%) + val (10%), test (10%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y_normalized, test_size=TEST_SPLIT, random_state=RANDOM_SEED
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=VALIDATION_SPLIT/(1-TEST_SPLIT), random_state=RANDOM_SEED
    )
    
    print(f"📊 Data split:")
    print(f"   Train: {len(X_train)} samples")
    print(f"   Val:   {len(X_val)} samples")
    print(f"   Test:  {len(X_test)} samples")
    
    # Train model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True
            )
        ]
    )
    
    print("\n✅ Training completed")
else:
    print("❌ Cannot train - no data loaded")

## 6. Evaluate Model

In [ ]:
if X is not None and len(X) > 0:
    # Evaluate on test set
    test_loss, test_mae, test_mse = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"📈 Test Set Results:")
    print(f"   Loss (MSE): {test_loss:.6f}")
    print(f"   MAE: {test_mae:.6f}")
    
    # Make predictions
    y_pred = model.predict(X_test, verbose=0).flatten()
    
    # Denormalize predictions to original scale [0, 100]
    y_pred_denorm = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    y_test_denorm = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    # Calculate denormalized metrics
    mae_denorm = np.mean(np.abs(y_pred_denorm - y_test_denorm))
    mse_denorm = np.mean((y_pred_denorm - y_test_denorm) ** 2)
    rmse_denorm = np.sqrt(mse_denorm)
    
    print(f"\n📊 Denormalized Metrics (Scale 0-100):")
    print(f"   MAE: {mae_denorm:.2f}")
    print(f"   RMSE: {rmse_denorm:.2f}")
    print(f"\n🎯 Sample predictions (first 5 test samples):")
    for i in range(min(5, len(y_test_denorm))):
        print(f"   True: {y_test_denorm[i]:.2f}, Predicted: {y_pred_denorm[i]:.2f}")

## 7. Visualize Training History

In [ ]:
if X is not None and len(X) > 0 and 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Loss plot
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].set_title('Model Loss Over Epochs')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # MAE plot
    axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].set_title('Model MAE Over Epochs')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Training history plotted")

## 8. Save Model

In [ ]:
# Save model
model_path = 'cnn_iqa_model.h5'
model.save(model_path)
print(f"✅ Model saved to {model_path}")

# Save scaler for later denormalization
import pickle
scaler_path = 'quality_scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler saved to {scaler_path}")

## 9. Test on Single Image Pair

In [ ]:
def predict_quality(orig_path, rescaled_path, model, scaler, patch_size=32):
    """
    Predict quality score for an image pair
    """
    try:
        paired = load_image_pair(orig_path, rescaled_path, patch_size)
        if paired is None:
            return None
        
        # Add batch dimension
        paired_batch = np.expand_dims(paired, axis=0)
        
        # Predict (returns normalized value)
        pred_normalized = model.predict(paired_batch, verbose=0)[0][0]
        
        # Denormalize to [0, 100]
        pred_quality = scaler.inverse_transform([[pred_normalized]])[0][0]
        
        return pred_quality
    except Exception as e:
        print(f"Error: {e}")
        return None


# Example: Test on first test sample
if X is not None and len(X_test) > 0:
    print("Test prediction on first test sample:")
    print(f"  True quality (normalized): {y_test[0]:.4f}")
    print(f"  True quality (0-100): {y_test_denorm[0]:.2f}")
    print(f"  Predicted quality (0-100): {y_pred_denorm[0]:.2f}")